In [ ]:
import json
import re
from copy import deepcopy
from pathlib import Path
from google.colab import files

## 1. Korpus hasil pemecah kalimat


In [ ]:
unggahan = files.upload()
nama_input = next(iter(unggahan))
data = json.loads(
    unggahan[nama_input].decode("utf-8-sig")
)

Saving korpus_kalimat.json to korpus_kalimat (2).json


## Baca kamus alias

In [ ]:
unggahan_alias = files.upload()
nama_alias = next(iter(unggahan_alias))

alias_input = json.loads(
    unggahan_alias[nama_alias].decode("utf-8-sig")
)

alias = {
    k.strip().lower(): v.strip().lower()
    for k, v in alias_input.items()
}

# Mendahulukan alias yang lebih panjang
daftar_alias = sorted(alias, key=len, reverse=True)

pola_alias = re.compile(
    r"(?<!\w)(?:"
    + "|".join(re.escape(nama) for nama in daftar_alias)
    + r")(?!\w)"
)


def normalisasi_alias(teks):
    teks = re.sub(r"\s+", " ", teks).strip().lower()
    return pola_alias.sub(
        lambda cocok: alias[cocok.group()],
        teks,
    )

Saving alias.json to alias (1).json


## Menerapkan alias pada teks pencarian

In [ ]:
hasil = deepcopy(data)
jumlah_berubah = 0
contoh_perubahan = []

for k in hasil["kalimat"]:
    sebelum = k["teks_cari"]
    sesudah = normalisasi_alias(k["teks_asli"])

    if sebelum != sesudah:
        jumlah_berubah += 1

        if len(contoh_perubahan) < 3:
            contoh_perubahan.append({
                "id": k["id_kalimat"],
                "sebelum": sebelum,
                "sesudah": sesudah,
            })

    k["teks_cari"] = sesudah

hasil.setdefault("metadata", {})
hasil["metadata"]["aturan_alias"] = alias
hasil["metadata"]["normalisasi_teks_cari"] = (
    "Huruf kecil, spasi rapi, dan normalisasi alias."
)

In [ ]:
# Simpan hasil normalisasi alias
nama_output = "korpus_kalimat_alias.json"

Path(nama_output).write_text(
    json.dumps(hasil, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Jumlah kalimat:", len(hasil["kalimat"]))
print("Jumlah aturan alias:", len(alias))
print("Kalimat yang berubah:", jumlah_berubah)

for contoh in contoh_perubahan:
    print("\nID:", contoh["id"])
    print("Sebelum:", contoh["sebelum"])
    print("Sesudah:", contoh["sesudah"])


Jumlah kalimat: 3140
Jumlah aturan alias: 17
Kalimat yang berubah: 140

ID: D01-B0070-K001
Sebelum: kelompok sutan syahrir di daerah jakarta dan jawa barat dengan cara menyamar sebagai pedagang nanas di sindanglaya.
Sesudah: kelompok sutan sjahrir di daerah jakarta dan jawa barat dengan cara menyamar sebagai pedagang nanas di sindanglaya.

ID: D01-B0073-K001
Sebelum: kelompok achmad subardjo, sudiro dan wikana.
Sesudah: kelompok achmad soebardjo, sudiro dan wikana.

ID: D01-B0217-K002
Sebelum: anggota-anggotanya antara lain soekarno, hatta, wahid hasyim, dan lain-lain.
Sesudah: anggota-anggotanya antara lain soekarno, hatta, wahid hasjim, dan lain-lain.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

files.download(nama_output)


Tersimpan: korpus_kalimat_alias.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>